# NB11 — Final Inference Pipeline (M1 demonstration, no training)
**Locked model:** M1 = U-Net-32 + VV/VH/DEM, `models/nb06_m1_s1_dem_best.pt`, thr 0.5, train-derived `processed/normalization_stats.json`. This notebook only *demonstrates* the reusable `src.inference` module on a real test chip. Test/Bolivia data are used for inference/description only — no tuning, no training.

In [1]:
from pathlib import Path
import json
ROOT = Path(r"C:/Users/Swarnim/Desktop/ML projects/tsao")
cfg = json.loads((ROOT / "models" / "final_model_config.json").read_text())
print(json.dumps(cfg, indent=2))
assert cfg["model_name"] == "M1" and cfg["threshold"] == 0.5
assert cfg["input_channel_names"] == ["VV_dB", "VH_dB", "DEM_m"]
assert (ROOT / cfg["checkpoint_path"]).exists() and (ROOT / cfg["normalization_stats_path"]).exists()
print("locked config OK")

{
  "architecture": "UNet-32",
  "bolivia_f1": 0.7738611132251559,
  "bolivia_iou": 0.6311365885004935,
  "bolivia_precision": 0.8898565897938848,
  "bolivia_recall": 0.6846189762802015,
  "checkpoint_path": "models/nb06_m1_s1_dem_best.pt",
  "input_channel_names": [
    "VV_dB",
    "VH_dB",
    "DEM_m"
  ],
  "input_channels": 3,
  "model_name": "M1",
  "normalization_stats_path": "processed/normalization_stats.json",
  "test_f1": 0.8023157199586112,
  "test_iou": 0.6698891637498191,
  "test_precision": 0.8184893811433855,
  "test_recall": 0.7867688691248693,
  "threshold": 0.5,
  "training_seed": 42
}
locked config OK


In [2]:
import sys
sys.path.insert(0, str(ROOT))
from src.inference import FloodPredictor, compute_metrics, THRESHOLD
pred = FloodPredictor()  # loads locked ckpt + train stats, eval mode, no training
print("device:", pred.device, "| threshold:", pred.threshold)
n = sum(p.numel() for p in pred.model.parameters())
assert n == 7763041, n
print(f"params: {n:,} (M1 locked)")

device: cuda | threshold: 0.5
params: 7,763,041 (M1 locked)


In [3]:
import numpy as np
CHIP = "Mekong_1443339"  # real existing test chip
d = np.load(ROOT / "processed" / "test" / f"{CHIP}.npz", allow_pickle=True)
X, y, vm = d["X"].astype(np.float32), d["y"].astype(np.uint8), d["valid_mask"].astype(bool)
print("X", X.shape, X.dtype, "| y", y.shape, "| valid frac:", round(float(vm.mean()), 4))
print("crs:", str(d["crs"]), "| transform:", np.asarray(d["transform"])[:6])
vv, vh, dem = X[0], X[1], X[2]  # M1 channels only; rainfall never touched

X (6, 512, 512) float32 | y (512, 512) | valid frac: 1.0
crs: EPSG:4326 | transform: [ 8.98315284e-05  0.00000000e+00  1.05877595e+02  0.00000000e+00
 -8.98315284e-05  1.23263230e+01]


In [4]:
out = pred.predict(vv, vh, dem, valid_mask=vm, transform=np.asarray(d["transform"]), crs=str(d["crs"]))
prob, mask = out["probability"], out["mask"]
print("probability", prob.shape, prob.dtype, f"[{prob.min():.4f},{prob.max():.4f}]")
print("mask", mask.shape, mask.dtype, "values:", sorted(np.unique(mask).tolist()))
print("valid px:", out["valid_pixel_count"], "| flooded px:", out["flooded_pixel_count"])
print("flood fraction:", round(out["flood_fraction"], 6))
assert prob.shape == mask.shape == (512, 512) and np.isfinite(prob).all()
assert set(np.unique(mask).tolist()) <= {0, 1} and (prob.min() >= 0 and prob.max() <= 1)

probability (512, 512) float32 [0.0004,1.0000]
mask (512, 512) uint8 values: [0, 1]
valid px: 262135 | flooded px: 187661
flood fraction: 0.715894


In [5]:
from src.inference.flood_predictor import pixel_area_map_m2, transform_to_affine
area_map = pixel_area_map_m2((512, 512), transform_to_affine(np.asarray(d["transform"])))
print("per-pixel area range: [%.1f, %.1f] m^2 (latitude-corrected WGS84, EPSG:4326 not a constant)" % (area_map.min(), area_map.max()))
gt_area_km2 = float(area_map[((y == 1) & vm)].sum() / 1e6)
pa = float(out["flooded_area_km2"])
print("GT flooded area:   %.4f km2" % gt_area_km2)
print("Pred flooded area: %.4f km2" % pa)
print("abs err: %.4f km2 | rel err: %.2f %% (descriptive only)" % (abs(pa - gt_area_km2), 100 * abs(pa - gt_area_km2) / gt_area_km2))

per-pixel area range: [97.1, 97.1] m^2 (latitude-corrected WGS84, EPSG:4326 not a constant)
GT flooded area:   18.7210 km2
Pred flooded area: 18.2236 km2
abs err: 0.4974 km2 | rel err: 2.66 % (descriptive only)


In [6]:
m = compute_metrics(mask, y, vm)
print("chip: IoU=%.4f F1=%.4f P=%.4f R=%.4f" % (m["iou"], m["f1"], m["precision"], m["recall"]))
print("locked test-aggregate reference: IoU=0.6699 F1=0.8023 (single-chip value differs by design)")

chip: IoU=0.9102 F1=0.9530 P=0.9660 R=0.9403
locked test-aggregate reference: IoU=0.6699 F1=0.8023 (single-chip value differs by design)


In [7]:
from src.inference import visualize_inference
fig, _ = visualize_inference(vv, vh, dem, prob, mask, gt=y, valid_mask=vm, title="M1 " + CHIP,
                             area_km2=out["flooded_area_km2"], flood_fraction=out["flood_fraction"])
vp = ROOT / "results" / "final_inference_examples" / (CHIP + "_NB11_panel.png")
fig.savefig(vp, dpi=120); print("saved", vp)
fig  # display inline

saved C:\Users\Swarnim\Desktop\ML projects\tsao\results\final_inference_examples\Mekong_1443339_NB11_panel.png


<Figure size 2400x450 with 7 Axes>

In [8]:
import time
from pathlib import Path as _P
t0 = time.time()
TP = FP = FN = TN = 0.0
for p in sorted((ROOT / "processed" / "test").glob("*.npz")):
    o = pred.predict_npz(p)
    c = compute_metrics(o["mask"], o["gt"], o["valid_mask"])
    TP += c["tp"]; FP += c["fp"]; FN += c["fn"]; TN += c["tn"]
iou = TP / (TP + FP + FN + 1e-9); pr = TP / (TP + FP + 1e-9); rc = TP / (TP + FN + 1e-9)
f1 = 2 * pr * rc / (pr + rc + 1e-9)
print("test aggregate (90 chips, %.0fs): IoU=%.4f F1=%.4f P=%.4f R=%.4f" % (time.time()-t0, iou, f1, pr, rc))
print("locked M1 reference: IoU=0.6699 F1=0.8023 P=0.8185 R=0.7868")
assert abs(iou - 0.6699) < 1e-3 and abs(f1 - 0.8023) < 1e-3, "reproduction FAILED - STOP"
print("REPRODUCTION PASS (tol 1e-3; observed diff ~1e-6 from AMP float)")

test aggregate (90 chips, 7s): IoU=0.6699 F1=0.8023 P=0.8185 R=0.7868
locked M1 reference: IoU=0.6699 F1=0.8023 P=0.8185 R=0.7868
REPRODUCTION PASS (tol 1e-3; observed diff ~1e-6 from AMP float)


## Conclusion
The reusable `src.inference` pipeline reproduces the locked M1 test result (IoU ≈ 0.6699, F1 ≈ 0.8023), estimates latitude-corrected flooded area, writes georeferenced GeoTIFFs, and exposes a CLI (`python -m src.inference --npz <chip> --output <dir>`). No retraining, no tuning, no new experiments. **NEXT STEP = T-SAO WEBSITE.**